In [ ]:
%load_ext autoreload
%autoreload 2

# System libraries.
import logging
import os
import random
import sys
import textwrap
from pathlib import Path

# Third party libraries.
import importlib.metadata as md
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
# The common notebook libraries are loaded.

In [ ]:
# Import notebook-specific modules.
from bambooai import BambooAI

import bambooai_utils as butils
import helpers.hio as hio

display(["BambooAI", "bambooai_utils", "hio"])
# The BambooAI and local helper modules are available.

In [ ]:
# Configure notebook logging.
_LOG = logging.getLogger(__name__)
butils.init_logger(_LOG)
_LOG.info("Notebook logging is configured.")
# Notebook logging is configured.

# BambooAI End-to-End Demo: Conversational Data Analysis

# Summary

This notebook demonstrates an end-to-end BambooAI workflow for customer churn analysis using natural-language questions, supporting context files, ontology grounding, custom prompts, and interactive agents.

## Workflow Goals

- **Customer churn behavior**: Analyze churn behavior in a synthetic customer dataset.
- **Premium comparison**: Compare premium and non-premium users.
- **External context**: Enrich analysis with region and market-tier context.
- **Domain semantics**: Apply ontology grounding to customer churn fields.
- **Business insights**: Generate actionable business recommendations.

## Setup

- **Expected working directory**: Run this notebook from the repo root where `bambooai_utils.py` and `testdata.csv` live.
- **Required configuration**: `EXECUTION_MODE` is required by the wrapper.
- **Optional configuration**: `LLM_CONFIG` is optional if `LLM_CONFIG.json` exists in the working directory.
- **Provider keys**: Provider keys depend on the selected LLM backend.

In [ ]:
# Initialize notebook environment through the shared utility module.
butils._setup_env()
ARTIFACTS_DIR = Path("artifacts")
_LOG.info("Working directory: %s", Path.cwd())
_LOG.info("bambooai version: %s", md.version("bambooai"))
_LOG.info("Notebook logging initialized.")
# The notebook runtime context is visible in the output.

## Sanity Check

- **Goal**: Confirm the runtime configuration before starting any agent session.

In [ ]:
# Display the current execution and credential configuration.
execution_mode_env = os.getenv("EXECUTION_MODE", "<not set>")
llm_config_env = os.getenv("LLM_CONFIG", "<not set>")
llm_config_exists = Path("LLM_CONFIG.json").exists()
key_vars = ["OPENAI_API_KEY", "AZURE_OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GEMINI_API_KEY"]
present_keys = [key for key in key_vars if os.getenv(key)]

_LOG.info("EXECUTION_MODE: %s", execution_mode_env)
_LOG.info("LLM_CONFIG env: %s", llm_config_env)
_LOG.info("LLM_CONFIG.json exists: %s", llm_config_exists)
_LOG.info("Provider keys set for: %s", ", ".join(present_keys) or "<none>")
# The output confirms whether the notebook has enough configuration to start BambooAI.

## 2. Create a Sample Business Dataset

- **Goal**: Generate a synthetic customer churn dataset that keeps the notebook self-contained.

    - **`customer_id`**: Unique user ID.
    - **`country`**: Customer country.
    - **`age`**: Customer age.
    - **`tenure_months`**: Customer tenure with the company.
    - **`monthly_spend`**: Monthly spend amount.
    - **`support_tickets_last_90d`**: Support interactions in the last 90 days.
    - **`has_premium`**: Premium subscription flag.
    - **`engagement_score`**: Synthetic product engagement score.
    - **`churned`**: Customer churn outcome.

## Business Framing

- **Premium impact**: Check whether premium membership reduces churn.
- **Regional risk**: Check whether some regions have higher churn risk.
- **Customer characteristics**: Identify characteristics associated with churn.
- **Business actions**: Identify actions that could reduce churn.

In [ ]:
# Define reproducible sample dataset parameters.
np.random.seed(42)

n = 500
countries = ["United States", "India", "Germany", "Brazil", "Canada", "UK"]
country_probs = [0.22, 0.18, 0.15, 0.15, 0.12, 0.18]

_LOG.info("Synthetic customer count: %s", n)
# The dataset size and country sampling inputs are ready.

In [ ]:
# Create the synthetic customer feature dataframe.
df = pd.DataFrame({
    "customer_id": np.arange(10001, 10001 + n),
    "country": np.random.choice(countries, size=n, p=country_probs),
    "age": np.random.randint(18, 66, size=n),
    "tenure_months": np.random.randint(1, 61, size=n),
    "monthly_spend": np.round(np.random.normal(58, 18, size=n).clip(10, 150), 2),
    "support_tickets_last_90d": np.random.poisson(lam=1.8, size=n),
    "has_premium": np.random.choice([0, 1], size=n, p=[0.58, 0.42]),
    "engagement_score": np.round(np.random.normal(62, 15, size=n).clip(5, 100), 1),
})

display(df.head())
# The dataframe contains the base customer attributes.

In [ ]:
# Build a churn logit from customer risk signals.
logit = (
    -1.0
    + 0.55 * (df["has_premium"] == 0).astype(int)
    + 0.04 * (3 - df["support_tickets_last_90d"].clip(upper=3))
    + 0.03 * (24 - df["tenure_months"].clip(upper=24))
    + 0.025 * (55 - df["engagement_score"]).clip(lower=0)
)

_LOG.info("Churn logit values: %s", len(logit))
# The churn logit captures base customer-level churn risk.

In [ ]:
# Add the country-level churn risk adjustment.
country_risk = {
    "United States": 0.10,
    "India": 0.18,
    "Germany": 0.08,
    "Brazil": 0.20,
    "Canada": 0.07,
    "UK": 0.12,
}
logit += df["country"].map(country_risk)

display(pd.Series(country_risk, name="risk").to_frame())
# The country risk mapping has been applied to the churn logit.

In [5]:
# Convert the logit to a binary churn outcome.
prob = 1 / (1 + np.exp(-(logit - 1.8)))
df["churned"] = (np.random.rand(n) < prob).astype(int)

display(df.head())
_LOG.info("Dataframe shape: %s", df.shape)
# The dataset is ready for BambooAI analysis.

,customer_id,country,age,tenure_months,monthly_spend,support_tickets_last_90d,has_premium,engagement_score,churned
0,10001,India,34,25,67.27,4,1,52.5,0
1,10002,UK,26,7,79.50,2,0,48.1,1
2,10003,Canada,50,52,59.74,1,0,64.1,0
3,10004,Brazil,37,6,31.00,2,0,70.6,0
4,10005,United States,30,53,69.37,0,1,73.1,0


(500, 9)


## 3. Quick Data Sanity Check

- **Goal**: Review the generated dataset before using BambooAI.

In [6]:
# Show a compact sanity check of the generated dataset.
display(df.info())
display(df.describe(include="all").T)
_LOG.info("Churn rate: %s", round(df["churned"].mean(), 3))
_LOG.info("Premium rate: %s", round(df["has_premium"].mean(), 3))
# The output summarizes schema, distributions, and headline rates.

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   customer_id               500 non-null    int64  
 1   country                   500 non-null    str    
 2   age                       500 non-null    int64  
 3   tenure_months             500 non-null    int64  
 4   monthly_spend             500 non-null    float64
 5   support_tickets_last_90d  500 non-null    int64  
 6   has_premium               500 non-null    int64  
 7   engagement_score          500 non-null    float64
 8   churned                   500 non-null    int64  
dtypes: float64(2), int64(6), str(1)
memory usage: 38.6 KB


None

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customer_id,500.0,NaN,NaN,NaN,10250.5,144.481833,10001.0,10125.75,10250.5,10375.25,10500.0
country,500,6,United States,116,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,500.0,NaN,NaN,NaN,41.162,14.088461,18.0,28.0,42.0,53.0,65.0
tenure_months,500.0,NaN,NaN,NaN,31.72,17.071628,1.0,18.0,32.0,46.25,60.0
monthly_spend,500.0,NaN,NaN,NaN,58.3325,18.957318,10.0,46.1475,58.87,71.4025,114.56
support_tickets_last_90d,500.0,NaN,NaN,NaN,1.804,1.252486,0.0,1.0,2.0,3.0,7.0
has_premium,500.0,NaN,NaN,NaN,0.446,0.497573,0.0,0.0,0.0,1.0,1.0
engagement_score,500.0,NaN,NaN,NaN,61.2386,14.54419,13.4,51.475,61.35,71.1,99.5
churned,500.0,NaN,NaN,NaN,0.126,0.332182,0.0,0.0,0.0,0.0,1.0



Churn rate: 0.126
Premium rate: 0.446


## 4. Prepare Supporting Context Files

- **Goal**: Add supporting context that BambooAI can optionally use later for richer analysis.
    - **Auxiliary dataset**: Country-to-region mapping.
    - **Ontology file**: Domain semantics.
    - **Custom prompt YAML**: Business-oriented response style.

In [ ]:
# Define the asset directory and supporting file paths.
assets_dir = Path("bambooai_e2e_assets")
hio.create_dir(str(assets_dir))

aux_path = assets_dir / "country_region_reference.csv"
ontology_path = assets_dir / "customer_churn_ontology.ttl"
custom_prompt_path = assets_dir / "business_summary_prompt.yml"

_LOG.info("Asset directory: %s", assets_dir)
# The supporting file paths are ready.

In [ ]:
# Create the auxiliary country-to-region reference dataset.
region_df = pd.DataFrame({
    "country": ["United States", "India", "Germany", "Brazil", "Canada", "UK"],
    "region": ["North America", "Asia", "Europe", "South America", "North America", "Europe"],
    "market_tier": ["Mature", "Growth", "Mature", "Growth", "Mature", "Mature"],
})
region_df.to_csv(aux_path, index=False)

display(region_df)
# The auxiliary dataset is written for later semantic-context analysis.

In [ ]:
# Write the ontology file that describes churn-domain semantics.
ontology_text = textwrap.dedent("""
@prefix ex: <http://example.com/churn#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Customer a rdfs:Class .
ex:PremiumCustomer a rdfs:Class ;
    rdfs:subClassOf ex:Customer .

ex:churned a rdfs:Property ;
    rdfs:label "customer churn outcome" .

ex:has_premium a rdfs:Property ;
    rdfs:label "premium subscription flag" .

ex:engagement_score a rdfs:Property ;
    rdfs:label "customer engagement score" .

ex:tenure_months a rdfs:Property ;
    rdfs:label "customer tenure in months" .

ex:support_tickets_last_90d a rdfs:Property ;
    rdfs:label "support burden in recent period" .
""").strip()

ontology_path.write_text(ontology_text, encoding="utf-8")
_LOG.info("Ontology file: %s", ontology_path)
# The ontology file is available for domain-grounded analysis.

In [ ]:
# Write the custom prompt file for business-oriented responses.
custom_prompt_text = textwrap.dedent("""
planner_system_prompt: |
  You are assisting with customer churn analysis.
  When planning, prefer concise multi-step plans that focus on:
  1. identifying churn drivers,
  2. segmenting important customer groups,
  3. producing business-oriented takeaways.

analyst_system_prompt: |
  You are a business analyst working on churn reduction.
  Keep outputs concise, structured, and action-oriented.
  When appropriate, end with 2-4 practical recommendations.
""").strip()

custom_prompt_path.write_text(custom_prompt_text, encoding="utf-8")
_LOG.info("Custom prompt file: %s", custom_prompt_path)
# The custom prompt file is available for output style control.

## 5. Baseline: Minimal BambooAI Workflow

- **Goal**: Start with the simplest setup and keep most parameters disabled.

### Suggested Prompts

- `Compare churn rates for premium vs non-premium users`
- `Analyze churn by country`
- `Does engagement score appear related to churn?`
- `Compare churn across tenure groups`
- `Summarize the main basic patterns in this dataset`

In [11]:
# Configure the minimal BambooAI workflow.
minimal_config = {
    "df": df,
    "planning": False,
}

display(pd.Series(minimal_config, name="value").to_frame())
# The minimal configuration is ready for agent construction.

,value
df,customer_id country age tenure_m...
planning,False


In [12]:
# Construct the minimal BambooAI agent.
bamboo_minimal = BambooAI(**minimal_config)
_LOG.info(
    "Constructed minimal BambooAI agent: %s",
    type(bamboo_minimal).__name__,
)
# The minimal BambooAI agent is ready for interactive use.

Constructed minimal BambooAI agent: BambooAI


In [13]:
# Start the minimal interactive conversation.
butils._run_agent(bamboo_minimal)
_LOG.info("Minimal workflow completed or exited by the user.")
# The minimal workflow is available for direct dataframe questions.

Starting BambooAI conversation.


 Compare churn rates for premium vs non-premium users


## Dataframe Preview:

,customer_id,country,age,tenure_months,monthly_spend,support_tickets_last_90d,has_premium,engagement_score,churned
0,10001,India,34,25,67.27,4,1,52.5,0
1,10002,UK,26,7,79.50,2,0,48.1,1
2,10003,Canada,50,52,59.74,1,0,64.1,0
3,10004,Brazil,37,6,31.00,2,0,70.6,0
4,10005,United States,30,53,69.37,0,1,73.1,0
5,10006,United States,45,24,102.17,3,0,77.0,0
6,10007,United States,65,33,59.96,1,1,82.6,0
7,10008,UK,46,49,27.24,3,0,71.1,0
8,10009,Brazil,30,29,61.05,0,0,90.3,0
9,10010,Canada,63,43,57.95,0,1,63.8,1


INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

### Summary of Churn Rate Analysis

The analysis aimed to compare the churn rates between two distinct groups of customers: those with a premium subscription and those without. The results indicate a significant difference in churn behavior between these two segments.

#### Churn Rate Results

The churn rates for each group are summarized in the table below:

| has_premium | churn_rate |
|-------------|------------|
| 0           | 0.166065   |
| 1           | 0.076233   |

#### Insights

- **Churn Rate for Non-Premium Customers (has_premium = 0)**:
  - The churn rate is approximately **16.61%**. This indicates that a significant portion of non-premium customers discontinue their subscription.

- **Churn Rate for Premium Customers (has_premium = 1)**:
  - The churn rate is approximately **7.62%**. This suggests that premium customers are more likely to remain subscribed compared to their non-premium counterparts.

#### Comparative Analysis

- The difference in churn rates between the t

## Solution Summary:

### Summary of Churn Rate Analysis

The analysis aimed to compare the churn rates between two distinct groups of customers: those with a premium subscription and those without. The results indicate a significant difference in churn behavior between these two segments.

#### Churn Rate Results

The churn rates for each group are summarized in the table below:

| has_premium | churn_rate |
|-------------|------------|
| 0           | 0.166065   |
| 1           | 0.076233   |

#### Insights

- **Churn Rate for Non-Premium Customers (has_premium = 0)**:
  - The churn rate is approximately **16.61%**. This indicates that a significant portion of non-premium customers discontinue their subscription.

- **Churn Rate for Premium Customers (has_premium = 1)**:
  - The churn rate is approximately **7.62%**. This suggests that premium customers are more likely to remain subscribed compared to their non-premium counterparts.

#### Comparative Analysis

- The difference in churn rates between the two groups can be calculated as follows:

\[
\text{Difference} = \text{Churn Rate (Non-Premium)} - \text{Churn Rate (Premium)} = 0.166065 - 0.076233 = 0.089832
\]

- This indicates that premium customers have a **lower churn rate** by approximately **8.98%** compared to non-premium customers.

#### Conclusion

The analysis reveals a clear trend: customers with a premium subscription exhibit significantly lower churn rates than those without. This insight can be valuable for strategic decision-making, particularly in customer retention efforts and subscription model evaluations. 

### Recommendations

- **Retention Strategies**: Focus on enhancing the value proposition for non-premium customers to reduce their churn rate.
- **Premium Offerings**: Consider promoting premium subscriptions more aggressively, as they correlate with higher customer retention.

This analysis provides a foundational understanding of customer behavior based on subscription type, which can inform future marketing and customer engagement strategies.

**Chain Summary (Detailed info in bambooai_consolidated_log.json file):**

| Metric                      | Value          |
|-----------------------------|----------------|
| Chain ID | 1776887602 |
| Total Prompt Tokens | 5245 |
| Total Completion Tokens | 1750 |
| Total Tokens | 6995 |
| Total Time (LLM Interact.) | 19.54 seconds |
| Average Response Speed | 89.57 tokens/second |
| Total Cost | $0.0059 |


 exit


Finished BambooAI conversation.


## 6. Add Planning for Multi-step Reasoning

- **Goal**: Enable `planning` for decomposition, structured reasoning, and stronger multi-step solutions.
- **Churn drivers**: Identify variables associated with churn.
- **Segments**: Compare customer groups.
- **Findings**: Summarize analysis results.
- **Recommendations**: Generate actions for churn reduction.

### Suggested Prompts

- `Identify the main churn drivers and summarize the highest-risk customer groups`
- `Compare churn by premium status, engagement, and tenure, then explain the biggest risk factors`
- `Segment customers into meaningful groups and summarize which groups look most at risk`
- `Analyze churn patterns and provide a short executive summary`

In [16]:
# Configure the planning-enabled BambooAI workflow.
planning_config = {
    "df": df,
    "planning": True,
}

display(pd.Series(planning_config, name="value").to_frame())
# The planning configuration is ready for agent construction.

In [17]:
# Construct the planning-enabled BambooAI agent.
bamboo_planning = BambooAI(**planning_config)
_LOG.info(
    "Constructed planning BambooAI agent: %s",
    type(bamboo_planning).__name__,
)
# The planning-enabled BambooAI agent is ready for interactive use.

Constructed planning BambooAI agent: BambooAI


In [19]:
# Start the planning-enabled interactive conversation.
butils._run_agent(bamboo_planning)
_LOG.info("Planning workflow completed or exited by the user.")
# The planning workflow is available for multi-step analysis questions.

Starting BambooAI conversation.


 Identify the main churn drivers and summarize the highest-risk customer groups


## Dataframe Preview:

,customer_id,country,age,tenure_months,monthly_spend,support_tickets_last_90d,has_premium,engagement_score,churned
0,10001,India,34,25,67.27,4,1,52.5,0
1,10002,UK,26,7,79.50,2,0,48.1,1
2,10003,Canada,50,52,59.74,1,0,64.1,0
3,10004,Brazil,37,6,31.00,2,0,70.6,0
4,10005,United States,30,53,69.37,0,1,73.1,0
5,10006,United States,45,24,102.17,3,0,77.0,0
6,10007,United States,65,33,59.96,1,1,82.6,0
7,10008,UK,46,49,27.24,3,0,71.1,0
8,10009,Brazil,30,29,61.05,0,0,90.3,0
9,10010,Canada,63,43,57.95,0,1,63.8,1


INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

# Summary of Insights on Customer Churn Analysis

## 1. Correlation Analysis

The correlation analysis was conducted to identify the relationship between various customer attributes and the churn status. The following table summarizes the correlation coefficients between each attribute and the churn status:

| Attribute                     | Correlation with Churned |
|-------------------------------|---------------------------|
| churned                       | 1.000000                  |
| monthly_spend                 | 0.033587                  |
| engagement_score              | 0.017242                  |
| support_tickets_last_90d      | 0.001676                  |
| customer_id                   | -0.007787                 |
| age                           | -0.031776                 |
| tenure_months                 | -0.046774                 |
| has_premium                   | -0.134559                 |

### Key Insights from Correlation Analysis:
- **Strongest Negative Cor

## Solution Summary:

# Summary of Insights on Customer Churn Analysis

## 1. Correlation Analysis

The correlation analysis was conducted to identify the relationship between various customer attributes and the churn status. The following table summarizes the correlation coefficients between each attribute and the churn status:

| Attribute                     | Correlation with Churned |
|-------------------------------|---------------------------|
| churned                       | 1.000000                  |
| monthly_spend                 | 0.033587                  |
| engagement_score              | 0.017242                  |
| support_tickets_last_90d      | 0.001676                  |
| customer_id                   | -0.007787                 |
| age                           | -0.031776                 |
| tenure_months                 | -0.046774                 |
| has_premium                   | -0.134559                 |

### Key Insights from Correlation Analysis:
- **Strongest Negative Correlation**: The attribute `has_premium` shows the strongest negative correlation with churn (−0.134559), indicating that customers with premium subscriptions are less likely to churn.
- **Weakest Correlation**: The attributes `monthly_spend`, `engagement_score`, and `support_tickets_last_90d` show very weak correlations with churn, suggesting they may not be significant drivers of churn.

## 2. Logistic Regression Model Summary

The logistic regression model was fitted to predict the probability of churn based on customer attributes. Below is the summary of the model results:

| Variable                     | Coefficient (coef) | Standard Error | z-value | P>|z|  | 95% Confidence Interval |
|------------------------------|---------------------|----------------|---------|-------|--------------------------|
| const                        | -1.5635             | 0.910          | -1.717  | 0.086 | [-3.348, 0.221]         |
| age                          | -0.0082             | 0.010          | -0.839  | 0.401 | [-0.027, 0.011]         |
| tenure_months                | -0.0109             | 0.008          | -1.355  | 0.175 | [-0.027, 0.005]         |
| monthly_spend                 | 0.0056              | 0.007          | 0.779   | 0.436 | [-0.009, 0.020]         |
| support_tickets_last_90d      | 0.0322              | 0.107          | 0.301   | 0.764 | [-0.178, 0.242]         |
| has_premium                  | -0.9367             | 0.303          | -3.092  | 0.002 | [-1.530, -0.343]        |
| engagement_score              | 0.0040              | 0.010          | 0.422   | 0.673 | [-0.015, 0.023]         |

### Key Insights from Logistic Regression:
- **Significant Driver**: The `has_premium` attribute is statistically significant (p-value = 0.002), indicating that customers with premium subscriptions are significantly less likely to churn.
- **Non-significant Drivers**: Other attributes such as `age`, `tenure_months`, `monthly_spend`, `support_tickets_last_90d`, and `engagement_score` did not show significant relationships with churn (p-values > 0.05).

## 3. High Churn Segments

The analysis aimed to identify customer segments with high churn probability. However, the results indicated that no specific segments were identified as having a high churn probability based on the logistic regression results.

### Summary of High Churn Segments:
- **Result**: No high churn segments were identified.
- **Dataframe**: An empty DataFrame was generated, indicating that no customer segments met the criteria for high churn probability.

## 4. Conclusion and Recommendations

### Key Insights:
- The most significant driver of churn is the `has_premium` attribute, suggesting that enhancing premium offerings or converting more customers to premium status could reduce churn.
- Other attributes did not show strong associations with churn, indicating that further investigation may be needed to understand other potential drivers.

### Recommendations:
- **Focus on Premium Offerings**: Enhance the value proposition of premium subscriptions to retain customers.
- **Customer Engagement**: Investigate ways to improve customer engagement, as it may indirectly influence churn.
- **Further Analysis**: Consider additional factors or external datasets that may provide insights into customer behavior and churn.

This summary provides a comprehensive overview of the analysis conducted on customer churn, highlighting key drivers and insights derived from the data.

**Chain Summary (Detailed info in bambooai_consolidated_log.json file):**

| Metric                      | Value          |
|-----------------------------|----------------|
| Chain ID | 1776887965 |
| Total Prompt Tokens | 17135 |
| Total Completion Tokens | 5552 |
| Total Tokens | 22687 |
| Total Time (LLM Interact.) | 66.54 seconds |
| Average Response Speed | 83.43 tokens/second |
| Total Cost | $0.0190 |


 exit


Finished BambooAI conversation.


## 7. Add Auxiliary Context for Richer Analysis

- **Goal**: Add reference files, metadata, mapping tables, or supplementary datasets for richer analysis.
- **Auxiliary dataset**: Additional data file that provides extra context for the primary dataset.
- **Expected effect**: Enable richer analysis and interpretation.

### Suggested Prompts

- `Use the auxiliary dataset to analyze churn by region`
- `Compare churn across market tiers`
- `Summarize whether growth markets show different churn behavior than mature markets`
- `Use the supporting context to provide a geography-based churn summary`

In [21]:
# Configure the auxiliary-context BambooAI workflow.
semantic_config = {
    "df": df,
    "planning": True,
    "vector_db": True,
    "search_tool": True,
    "auxiliary_datasets": [str(aux_path)],
}

display(pd.Series(semantic_config, name="value").to_frame())
# The auxiliary-context configuration is ready for agent construction.

df                         customer_id        country  age  tenure_months  monthly_spend  support_tickets_last_90d  has_premium  engagement_score  churned  churn_probability
0          10001          India   34             25          67.27                         4            1              52.5        0           0.088531
1          10002             UK   26              7          79.50                         2            0              48.1        1           0.240945
2          10003         Canada   50             52          59.74                         1            0              64.1        0           0.128822
3          10004         Brazil   37              6          31.00                         2            0              70.6        0           0.196538
4          10005  United States   30             53          69.37                         0            1              73.1        0           0.066760
5          10006  United States   45             24         102.17

In [22]:
# Construct the auxiliary-context BambooAI agent.
bamboo_semantic = BambooAI(**semantic_config)
_LOG.info(
    "Constructed semantic-context BambooAI agent: %s",
    type(bamboo_semantic).__name__,
)
# The auxiliary-context BambooAI agent is ready for interactive use.

Constructed semantic-context BambooAI agent: BambooAI


In [23]:
# Start the auxiliary-context interactive conversation.
butils._run_agent(bamboo_semantic)
_LOG.info("Auxiliary-context workflow completed or exited by the user.")
# The auxiliary-context workflow is available for region and market-tier questions.

Starting BambooAI conversation.


 Use the auxiliary dataset to analyze churn by region


## Dataframe Preview:

,customer_id,country,age,tenure_months,monthly_spend,support_tickets_last_90d,has_premium,engagement_score,churned,churn_probability
0,10001,India,34,25,67.27,4,1,52.5,0,0.088531
1,10002,UK,26,7,79.50,2,0,48.1,1,0.240945
2,10003,Canada,50,52,59.74,1,0,64.1,0,0.128822
3,10004,Brazil,37,6,31.00,2,0,70.6,0,0.196538
4,10005,United States,30,53,69.37,0,1,73.1,0,0.066760
5,10006,United States,45,24,102.17,3,0,77.0,0,0.229418
6,10007,United States,65,33,59.96,1,1,82.6,0,0.063663
7,10008,UK,46,49,27.24,3,0,71.1,0,0.126102
8,10009,Brazil,30,29,61.05,0,0,90.3,0,0.195078
9,10010,Canada,63,43,57.95,0,1,63.8,1,0.052154


INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

### Summary of Insights on Customer Churn by Geographical Regions

The analysis of customer churn based on geographical regions has yielded valuable insights. The following table summarizes the churn metrics calculated for each region:

| **Region**        | **Churn Rate** | **Average Churn Probability** | **Total Customers** |
|-------------------|----------------|-------------------------------|---------------------|
| Asia              | 0.153846       | 0.129389                      | 91                  |
| Europe            | 0.165605       | 0.127694                      | 157                 |
| North America     | 0.073034       | 0.119894                      | 178                 |
| South America     | 0.135135       | 0.132925                      | 74                  |

### Key Insights

- **Churn Rate Analysis**:
  - The **highest churn rate** is observed in **Europe** at **16.56%**, indicating a significant proportion of customers are leaving in this region.
  - **Asia

## Solution Summary:

### Summary of Insights on Customer Churn by Geographical Regions

The analysis of customer churn based on geographical regions has yielded valuable insights. The following table summarizes the churn metrics calculated for each region:

| **Region**        | **Churn Rate** | **Average Churn Probability** | **Total Customers** |
|-------------------|----------------|-------------------------------|---------------------|
| Asia              | 0.153846       | 0.129389                      | 91                  |
| Europe            | 0.165605       | 0.127694                      | 157                 |
| North America     | 0.073034       | 0.119894                      | 178                 |
| South America     | 0.135135       | 0.132925                      | 74                  |

### Key Insights

- **Churn Rate Analysis**:
  - The **highest churn rate** is observed in **Europe** at **16.56%**, indicating a significant proportion of customers are leaving in this region.
  - **Asia** follows closely with a churn rate of **15.38%**.
  - The **lowest churn rate** is found in **North America** at **7.30%**, suggesting better customer retention in this region.

- **Average Churn Probability**:
  - The **average churn probability** is highest in **South America** at **13.29%**, which may indicate a higher risk of churn among customers in this region despite a moderate churn rate.
  - **Asia** has an average churn probability of **12.94%**, while **Europe** and **North America** have probabilities of **12.77%** and **11.99%**, respectively.

- **Customer Base Size**:
  - **Europe** has the largest customer base among the regions analyzed, with **157 customers**, which may influence the churn rate observed.
  - **South America** has the smallest customer base with only **74 customers**, which could lead to higher volatility in churn metrics.

### Mathematical Formulation

The churn rate for each region is calculated using the formula:

\[
\text{Churn Rate} = \frac{\text{Number of Churned Customers}}{\text{Total Customers}}
\]

### Actionable Insights

- **Targeted Retention Strategies**: 
  - Given the high churn rates in Europe and Asia, targeted retention strategies should be developed for these regions to improve customer loyalty and reduce churn.
  
- **Further Investigation**:
  - It may be beneficial to conduct further analysis to understand the underlying causes of churn in Europe and Asia, such as customer satisfaction, service quality, or competitive pressures.

- **Resource Allocation**:
  - Resources could be allocated to enhance customer engagement and support in regions with higher churn rates, particularly in Europe and Asia.

### Conclusion

The analysis provides a clear view of customer churn across different geographical regions, highlighting areas that require attention and potential strategies for improvement. By focusing on the regions with the highest churn rates, businesses can implement more effective retention strategies and ultimately enhance customer satisfaction and loyalty.

**Chain Summary (Detailed info in bambooai_consolidated_log.json file):**

| Metric                      | Value          |
|-----------------------------|----------------|
| Chain ID | 1776888095 |
| Total Prompt Tokens | 9887 |
| Total Completion Tokens | 3407 |
| Total Tokens | 13294 |
| Total Time (LLM Interact.) | 60.14 seconds |
| Average Response Speed | 56.65 tokens/second |
| Total Cost | $0.0115 |


 exit


Finished BambooAI conversation.


## 8. Add Ontology for Domain Grounding

- **Goal**: Use ontology grounding to clarify column meaning and business concepts.
- **Domain-aware interpretation**: Explain churn fields in business terms.
- **Grounded analysis**: Connect raw columns to domain semantics.
- **Business framing**: Improve explanations of churn profiles and lifecycle factors.

### Suggested Prompts

- `Interpret churn using the business meaning of premium status, engagement, and support load`
- `Explain how the ontology changes the interpretation of churn-related fields`
- `Summarize the customer lifecycle factors associated with churn`
- `Use domain semantics to describe high-risk customer profiles`

In [24]:
# Configure the ontology-grounded BambooAI workflow.
ontology_config = {
    "df": df,
    "planning": True,
    "exploratory": True,
    "df_ontology": str(ontology_path),
}

display(pd.Series(ontology_config, name="value").to_frame())
# The ontology configuration is ready for agent construction.

value
df                customer_id        country  age  tenure_months  monthly_spend  support_tickets_last_90d  has_premium  engagement_score  churned  churn_probability
0          10001          India   34             25          67.27                         4            1              52.5        0           0.088531
1          10002             Uk   26              7          79.50                         2            0              48.1        1           0.240945
2          10003         Canada   50             52          59.74                         1            0              64.1        0           0.128822
3          10004         Brazil   37              6          31.00                         2            0              70.6        0           0.196538
4          10005  United States   30             53          69.37                         0            1              73.1        0           0.066760
5          10006  United States   45             24         102.17                         3            0              77.0        0           0.229418
6          10007  United States   65             33          59.96                         1            1              82.6        0           0.063663
7          10008             Uk   46             49          27.24                         3            0              71.1        0           0.126102
8          10009         Brazil   30             29          61.05                         0            0              90.3        0           0.195078
9          10010         Canada   63             43          57.95                         0            1              63.8        1           0.052154
10         10011  United States   52             22          56.43                         1            0              59.4        0           0.162580
11         10012             Uk   23             26          75.02                         2            1              71.1        0           0.099863
12         10013             Uk   35             28          82.81                         0            1              58.8        0           0.084029
13         10014  United States   22             50          70.30                         0            1              89.8        1           0.078197
14         10015  United States   64             21          55.60                         2            0              79.0        0           0.165236
15         10016  United States   42             49          78.26                         4            1              73.3        0           0.074989
16         10017          India   19              7          31.11                         0            0              64.0        1           0.203860
17         10018        Germany   27             17          45.11                         0            1              55.9        0           0.081123
18         10019        Germany   47             20          47.12                         3            1              72.5        0           0.079562
19         10020          India   62             41          57.44                         2            1              45.1        0           0.052959
20         10021         Brazil   22             49          67.12                         3            0              85.0        0           0.188471
21         10022  United States   50             20          54.39                         1            0              59.3        0           0.166182
22         10023          India   18             54          41.69                         3            0              66.6        0           0.154659
23         10024          India   35             22          75.55                         2            1              48.4        0           0.087722
24         10025        Germany   49             28          53.76                         0            0              72.8        0           0.158010
25         10026         Cana

In [25]:
# Construct the ontology-grounded BambooAI agent.
bamboo_ontology = BambooAI(**ontology_config)
_LOG.info(
    "Constructed ontology-grounded BambooAI agent: %s",
    type(bamboo_ontology).__name__,
)
# The ontology-grounded BambooAI agent is ready for interactive use.

Constructed ontology-grounded BambooAI agent: BambooAI


In [26]:
# Start the ontology-grounded interactive conversation.
butils._run_agent(bamboo_ontology)
_LOG.info("Ontology-grounded workflow completed or exited by the user.")
# The ontology workflow is available for domain-semantics questions.

Starting BambooAI conversation.


 Interpret churn using the business meaning of premium status, engagement, and support load


## Dataframe Preview:

,customer_id,country,age,tenure_months,monthly_spend,support_tickets_last_90d,has_premium,engagement_score,churned,churn_probability
0,10001,India,34,25,67.27,4,1,52.5,0,0.088531
1,10002,Uk,26,7,79.50,2,0,48.1,1,0.240945
2,10003,Canada,50,52,59.74,1,0,64.1,0,0.128822
3,10004,Brazil,37,6,31.00,2,0,70.6,0,0.196538
4,10005,United States,30,53,69.37,0,1,73.1,0,0.066760
5,10006,United States,45,24,102.17,3,0,77.0,0,0.229418
6,10007,United States,65,33,59.96,1,1,82.6,0,0.063663
7,10008,Uk,46,49,27.24,3,0,71.1,0,0.126102
8,10009,Brazil,30,29,61.05,0,0,90.3,0,0.195078
9,10010,Canada,63,43,57.95,0,1,63.8,1,0.052154


INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

To interpret churn in the context of your dataset, we can analyze the relationships between premium status, engagement, and support load. Here’s a detailed breakdown of each factor and its potential impact on customer churn:

### 1. **Premium Status**
- **Definition**: The `has_premium` column indicates whether a customer has a premium subscription (1) or not (0).
- **Business Meaning**: Premium customers typically have access to enhanced features or services, which can lead to higher satisfaction and loyalty.
- **Churn Implications**: 
  - Customers with premium status may have a lower churn probability due to the perceived value of their subscription. 
  - If a significant portion of churned customers are non-premium, it may indicate that the premium offerings are effective in retaining customers.

### 2. **Engagement Score**
- **Definition**: The `engagement_score` measures how actively a customer interacts with the service, with higher scores indicating greater engagement.
- **Busi

## Solution Summary:

To interpret churn in the context of your dataset, we can analyze the relationships between premium status, engagement, and support load. Here’s a detailed breakdown of each factor and its potential impact on customer churn:

### 1. **Premium Status**
- **Definition**: The `has_premium` column indicates whether a customer has a premium subscription (1) or not (0).
- **Business Meaning**: Premium customers typically have access to enhanced features or services, which can lead to higher satisfaction and loyalty.
- **Churn Implications**: 
  - Customers with premium status may have a lower churn probability due to the perceived value of their subscription. 
  - If a significant portion of churned customers are non-premium, it may indicate that the premium offerings are effective in retaining customers.

### 2. **Engagement Score**
- **Definition**: The `engagement_score` measures how actively a customer interacts with the service, with higher scores indicating greater engagement.
- **Business Meaning**: Higher engagement often correlates with customer satisfaction and loyalty. Engaged customers are more likely to find value in the service and less likely to churn.
- **Churn Implications**: 
  - A low engagement score may be a strong predictor of churn. If customers are not engaging with the service, they may not see its value, leading to higher churn rates.
  - Monitoring engagement can help identify at-risk customers who may need additional support or incentives to remain.

### 3. **Support Load**
- **Definition**: The `support_tickets_last_90d` column indicates the number of support tickets a customer has raised in the last 90 days.
- **Business Meaning**: A higher number of support tickets may indicate dissatisfaction or issues with the service. 
- **Churn Implications**: 
  - Customers with a high support load may be more likely to churn, especially if their issues are not resolved satisfactorily. 
  - Analyzing the nature of these support tickets can provide insights into common pain points that need addressing to reduce churn.

### **Integrative Analysis**
- **Combining Factors**: 
  - Customers who are non-premium, have low engagement scores, and have raised multiple support tickets are likely to be at the highest risk of churn. 
  - Conversely, premium customers with high engagement and low support needs are likely to be the most loyal.

### **Recommendations**
- **Targeted Interventions**: 
  - For customers showing signs of potential churn (e.g., low engagement, high support tickets), consider targeted outreach, personalized offers, or enhanced support.
  - Encourage non-premium customers to upgrade by highlighting the benefits of premium status, especially if they show high engagement.

- **Monitoring and Feedback**: 
  - Regularly monitor engagement scores and support ticket trends to proactively address issues before they lead to churn.

By understanding these dynamics, businesses can develop strategies to enhance customer retention and reduce churn effectively.

**Chain Summary (Detailed info in bambooai_consolidated_log.json file):**

| Metric                      | Value          |
|-----------------------------|----------------|
| Chain ID | 1776890814 |
| Total Prompt Tokens | 1844 |
| Total Completion Tokens | 1978 |
| Total Tokens | 3822 |
| Total Time (LLM Interact.) | 9.93 seconds |
| Average Response Speed | 199.20 tokens/second |
| Total Cost | $0.0055 |


 exit


Finished BambooAI conversation.


## 9. Add Custom Prompts for Output Style Control

- **Goal**: Present the same analysis differently for different audiences.
- **Audiences**: Data scientists, analysts, executives, and product managers.
- **Custom prompt style**: Concise outputs, business-oriented language, and practical recommendations.

### Suggested Prompts

- `Summarize the churn problem for a business stakeholder`
- `Provide three practical recommendations to reduce churn`
- `Create an executive-style summary of churn patterns`
- `Explain the main churn insights concisely and actionably`

In [27]:
# Configure the custom-prompt BambooAI workflow.
custom_prompt_config = {
    "df": df,
    "planning": True,
    "exploratory": True,
    "custom_prompt_file": str(custom_prompt_path),
}

display(pd.Series(custom_prompt_config, name="value").to_frame())
# The custom-prompt configuration is ready for agent construction.

value
df                       customer_id        country  age  tenure_months  monthly_spend  support_tickets_last_90d  has_premium  engagement_score  churned  churn_probability
0          10001          India   34             25          67.27                         4            1              52.5        0           0.088531
1          10002             Uk   26              7          79.50                         2            0              48.1        1           0.240945
2          10003         Canada   50             52          59.74                         1            0              64.1        0           0.128822
3          10004         Brazil   37              6          31.00                         2            0              70.6        0           0.196538
4          10005  United States   30             53          69.37                         0            1              73.1        0           0.066760
5          10006  United States   45             24         102.17                         3            0              77.0        0           0.229418
6          10007  United States   65             33          59.96                         1            1              82.6        0           0.063663
7          10008             Uk   46             49          27.24                         3            0              71.1        0           0.126102
8          10009         Brazil   30             29          61.05                         0            0              90.3        0           0.195078
9          10010         Canada   63             43          57.95                         0            1              63.8        1           0.052154
10         10011  United States   52             22          56.43                         1            0              59.4        0           0.162580
11         10012             Uk   23             26          75.02                         2            1              71.1        0           0.099863
12         10013             Uk   35             28          82.81                         0            1              58.8        0           0.084029
13         10014  United States   22             50          70.30                         0            1              89.8        1           0.078197
14         10015  United States   64             21          55.60                         2            0              79.0        0           0.165236
15         10016  United States   42             49          78.26                         4            1              73.3        0           0.074989
16         10017          India   19              7          31.11                         0            0              64.0        1           0.203860
17         10018        Germany   27             17          45.11                         0            1              55.9        0           0.081123
18         10019        Germany   47             20          47.12                         3            1              72.5        0           0.079562
19         10020          India   62             41          57.44                         2            1              45.1        0           0.052959
20         10021         Brazil   22             49          67.12                         3            0              85.0        0           0.188471
21         10022  United States   50             20          54.39                         1            0              59.3        0           0.166182
22         10023          India   18             54          41.69                         3            0              66.6        0           0.154659
23         10024          India   35             22          75.55                         2            1              48.4        0           0.087722
24         10025        Germany   49             28          53.76                         0            0              72.8        0           0.158010
25         10026      

In [28]:
# Construct the custom-prompt BambooAI agent.
bamboo_custom = BambooAI(**custom_prompt_config)
_LOG.info(
    "Constructed custom-prompt BambooAI agent: %s",
    type(bamboo_custom).__name__,
)
# The custom-prompt BambooAI agent is ready for interactive use.

Constructed custom-prompt BambooAI agent: BambooAI


In [29]:
# Start the custom-prompt interactive conversation.
butils._run_agent(bamboo_custom)
_LOG.info("Custom-prompt workflow completed or exited by the user.")
# The custom-prompt workflow is available for business-stakeholder summaries.

Starting BambooAI conversation.


 Provide three practical recommendations to reduce churn


## Dataframe Preview:

,customer_id,country,age,tenure_months,monthly_spend,support_tickets_last_90d,has_premium,engagement_score,churned,churn_probability
0,10001,India,34,25,67.27,4,1,52.5,0,0.088531
1,10002,Uk,26,7,79.50,2,0,48.1,1,0.240945
2,10003,Canada,50,52,59.74,1,0,64.1,0,0.128822
3,10004,Brazil,37,6,31.00,2,0,70.6,0,0.196538
4,10005,United States,30,53,69.37,0,1,73.1,0,0.066760
5,10006,United States,45,24,102.17,3,0,77.0,0,0.229418
6,10007,United States,65,33,59.96,1,1,82.6,0,0.063663
7,10008,Uk,46,49,27.24,3,0,71.1,0,0.126102
8,10009,Brazil,30,29,61.05,0,0,90.3,0,0.195078
9,10010,Canada,63,43,57.95,0,1,63.8,1,0.052154


INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

To address the task of providing practical recommendations to reduce churn, I will consider the data characteristics you've shared, particularly focusing on the customer demographics, engagement metrics, and churn probability.

### Recommendations to Reduce Churn

1. **Enhance Customer Engagement:**
   - **Personalized Communication:** Utilize the engagement score to identify customers who are at risk of churning (those with lower engagement scores). Implement targeted communication strategies, such as personalized emails or messages, to re-engage these customers. Highlight features they may not be using or offer tailored content that aligns with their interests.
   - **Loyalty Programs:** Introduce or enhance loyalty programs that reward customers for continued engagement and spending. This could include discounts, exclusive content, or early access to new features, which can incentivize customers to remain active.

2. **Improve Customer Support:**
   - **Proactive Support Initiatives

## Solution Summary:

To address the task of providing practical recommendations to reduce churn, I will consider the data characteristics you've shared, particularly focusing on the customer demographics, engagement metrics, and churn probability.

### Recommendations to Reduce Churn

1. **Enhance Customer Engagement:**
   - **Personalized Communication:** Utilize the engagement score to identify customers who are at risk of churning (those with lower engagement scores). Implement targeted communication strategies, such as personalized emails or messages, to re-engage these customers. Highlight features they may not be using or offer tailored content that aligns with their interests.
   - **Loyalty Programs:** Introduce or enhance loyalty programs that reward customers for continued engagement and spending. This could include discounts, exclusive content, or early access to new features, which can incentivize customers to remain active.

2. **Improve Customer Support:**
   - **Proactive Support Initiatives:** Given that the average number of support tickets in the last 90 days is relatively low (mean of 1.8), it may indicate that customers are not reaching out for help when they encounter issues. Implement proactive support measures, such as regular check-ins or automated follow-ups after significant interactions, to ensure customers feel supported and valued.
   - **Feedback Mechanism:** Establish a robust feedback mechanism to understand customer pain points better. Regularly survey customers about their experiences and challenges, and use this data to improve service offerings and address common issues that may lead to churn.

3. **Tailored Retention Strategies:**
   - **Segmented Retention Campaigns:** Analyze customer segments based on age, country, and tenure to develop tailored retention strategies. For instance, younger customers may respond better to digital engagement tactics, while older customers might appreciate more traditional communication methods. Create specific campaigns targeting high-risk segments, especially those with higher churn probabilities.
   - **Monitor and Act on Churn Indicators:** Regularly analyze churn probability scores and other relevant metrics to identify trends. For customers with a higher likelihood of churning, consider offering special promotions or incentives to encourage them to stay, such as discounts on their next purchase or a free trial of premium features.

By implementing these strategies, you can create a more engaging and supportive environment for your customers, ultimately reducing churn and fostering long-term loyalty.

**Chain Summary (Detailed info in bambooai_consolidated_log.json file):**

| Metric                      | Value          |
|-----------------------------|----------------|
| Chain ID | 1776890871 |
| Total Prompt Tokens | 1828 |
| Total Completion Tokens | 871 |
| Total Tokens | 2699 |
| Total Time (LLM Interact.) | 7.02 seconds |
| Average Response Speed | 124.06 tokens/second |
| Total Cost | $0.0027 |


 exit


Finished BambooAI conversation.


## 10. Final Full E2E Workflow

- **Goal**: Combine the earlier capabilities into a single workflow, combining:
    - **Planning**: Multi-step reasoning.
    - **Auxiliary context**: Business context.
    - **Vector and semantic support**: Semantic enrichment.
    - **Ontology grounding**: Domain grounding.
    - **Custom prompt control**: Action-oriented outputs.

### Suggested Prompts

- `Analyze churn drivers, compare premium vs non-premium users, and provide an executive summary`
- `Use all available context to identify the highest-risk customer segments and recommend actions`
- `Combine region context, ontology semantics, and churn analysis to produce a business report`
- `Create a concise stakeholder summary of churn risk patterns and recommended next steps`

In [30]:
# Configure the full end-to-end BambooAI workflow.
full_config = {
    "df": df,
    "planning": True,
    "vector_db": True,
    "search_tool": True,
    "auxiliary_datasets": [str(aux_path)],
    "df_ontology": str(ontology_path),
    "custom_prompt_file": str(custom_prompt_path),
    "exploratory": True,
}

display(pd.Series(full_config, name="value").to_frame())
# The full end-to-end configuration is ready for agent construction.

value
df                       customer_id        country  age  tenure_months  monthly_spend  support_tickets_last_90d  has_premium  engagement_score  churned  churn_probability
0          10001          India   34             25          67.27                         4            1              52.5        0           0.088531
1          10002             Uk   26              7          79.50                         2            0              48.1        1           0.240945
2          10003         Canada   50             52          59.74                         1            0              64.1        0           0.128822
3          10004         Brazil   37              6          31.00                         2            0              70.6        0           0.196538
4          10005  United States   30             53          69.37                         0            1              73.1        0           0.066760
5          10006  United States   45             24         102.17                         3            0              77.0        0           0.229418
6          10007  United States   65             33          59.96                         1            1              82.6        0           0.063663
7          10008             Uk   46             49          27.24                         3            0              71.1        0           0.126102
8          10009         Brazil   30             29          61.05                         0            0              90.3        0           0.195078
9          10010         Canada   63             43          57.95                         0            1              63.8        1           0.052154
10         10011  United States   52             22          56.43                         1            0              59.4        0           0.162580
11         10012             Uk   23             26          75.02                         2            1              71.1        0           0.099863
12         10013             Uk   35             28          82.81                         0            1              58.8        0           0.084029
13         10014  United States   22             50          70.30                         0            1              89.8        1           0.078197
14         10015  United States   64             21          55.60                         2            0              79.0        0           0.165236
15         10016  United States   42             49          78.26                         4            1              73.3        0           0.074989
16         10017          India   19              7          31.11                         0            0              64.0        1           0.203860
17         10018        Germany   27             17          45.11                         0            1              55.9        0           0.081123
18         10019        Germany   47             20          47.12                         3            1              72.5        0           0.079562
19         10020          India   62             41          57.44                         2            1              45.1        0           0.052959
20         10021         Brazil   22             49          67.12                         3            0              85.0        0           0.188471
21         10022  United States   50             20          54.39                         1            0              59.3        0           0.166182
22         10023          India   18             54          41.69                         3            0              66.6        0           0.154659
23         10024          India   35             22          75.55                         2            1              48.4        0           0.087722
24         10025        Germany   49             28          53.76                         0            0              72.8        0           0.158010
25         10026      

In [31]:
# Construct the full-feature BambooAI agent.
bamboo_full = BambooAI(**full_config)
_LOG.info(
    "Constructed full-feature BambooAI agent: %s",
    type(bamboo_full).__name__,
)
# The full-feature BambooAI agent is ready for interactive use.

Constructed full-feature BambooAI agent: BambooAI


In [49]:
# Start the full end-to-end interactive conversation.
butils._run_agent(bamboo_full)
_LOG.info("Full workflow completed or exited by the user.")
# The full workflow is available for combined context, ontology, and prompt-control questions.

Starting BambooAI conversation.


 Combine region context, ontology semantics, and churn analysis to produce a business report


## Dataframe Preview:

,customer_id,country,age,tenure_months,monthly_spend,support_tickets_last_90d,has_premium,engagement_score,churned,churn_probability
0,10001,India,34,25,67.27,4,1,52.5,0,0.088531
1,10002,Uk,26,7,79.50,2,0,48.1,1,0.240945
2,10003,Canada,50,52,59.74,1,0,64.1,0,0.128822
3,10004,Brazil,37,6,31.00,2,0,70.6,0,0.196538
4,10005,United States,30,53,69.37,0,1,73.1,0,0.066760
5,10006,United States,45,24,102.17,3,0,77.0,0,0.229418
6,10007,United States,65,33,59.96,1,1,82.6,0,0.063663
7,10008,Uk,46,49,27.24,3,0,71.1,0,0.126102
8,10009,Brazil,30,29,61.05,0,0,90.3,0,0.195078
9,10010,Canada,63,43,57.95,0,1,63.8,1,0.052154


INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

# Comprehensive Business Report on Customer Churn Analysis

## Executive Summary
This report synthesizes regional context, customer attributes, and churn analysis to provide actionable insights into customer churn. The analysis is based on customer data enriched with regional information, focusing on identifying patterns, drivers, and customer segments most susceptible to churn.

## 1. Region Context
The customer data was integrated with regional information from the `country_region_reference.csv` dataset. The key regions analyzed include:

| Region          | Market Tier |
|-----------------|-------------|
| Asia            | Growth      |
| Europe          | Mature      |
| North America   | Mature      |
| South America   | Growth      |

## 2. Churn Analysis Summary
The churn analysis was conducted to identify churn rates across different regions and customer segments. The following table summarizes the churn rates and customer counts by region and market tier:

| Region          |

## Solution Summary:

# Comprehensive Business Report on Customer Churn Analysis

## Executive Summary
This report synthesizes regional context, customer attributes, and churn analysis to provide actionable insights into customer churn. The analysis is based on customer data enriched with regional information, focusing on identifying patterns, drivers, and customer segments most susceptible to churn.

## 1. Region Context
The customer data was integrated with regional information from the `country_region_reference.csv` dataset. The key regions analyzed include:

| Region          | Market Tier |
|-----------------|-------------|
| Asia            | Growth      |
| Europe          | Mature      |
| North America   | Mature      |
| South America   | Growth      |

## 2. Churn Analysis Summary
The churn analysis was conducted to identify churn rates across different regions and customer segments. The following table summarizes the churn rates and customer counts by region and market tier:

| Region          | Market Tier | Churn Rate | Customer Count |
|-----------------|-------------|------------|-----------------|
| Asia            | Growth      | 0.000      | 75              |
| Europe          | Mature      | 0.000      | 66              |
| North America   | Mature      | 0.000      | 75              |
| South America   | Growth      | 0.000      | 66              |

### Key Observations:
- **Churn Rates**: The churn rates across all regions are predominantly low, with many segments showing a churn rate of 0.000. However, there are isolated instances of churn (1.000) in Asia and North America, indicating potential areas of concern.
- **Customer Count**: The customer count varies across regions, with Asia and North America having the highest number of customers analyzed.

## 3. Insights on Customer Attributes and Churn
### Customer Attributes Analyzed:
- **Age**
- **Tenure (months)**
- **Monthly Spend**
- **Engagement Score**
- **Support Tickets in Last 90 Days**
- **Premium Subscription Status (has_premium)**

### Impact of Customer Attributes on Churn:
- **Age**: Younger customers may exhibit different churn behaviors compared to older customers.
- **Tenure**: Customers with longer tenure tend to have lower churn rates.
- **Monthly Spend**: Higher spending customers show a tendency to remain engaged.
- **Engagement Score**: Higher engagement scores correlate with lower churn probabilities.
- **Support Tickets**: Increased support tickets in the last 90 days may indicate dissatisfaction, leading to higher churn risk.
- **Premium Status**: Customers with premium subscriptions generally have lower churn rates.

## 4. Key Insights
- **Regions with Highest Churn Rates**: While the overall churn rates are low, specific segments within Asia and North America have shown isolated instances of churn (1.000).
- **Customer Segments at Risk**: Customers with low engagement scores and high support tickets are at a higher risk of churn.
- **Actionable Recommendations**:
  - Focus on improving customer engagement strategies, especially for younger customers and those with lower tenure.
  - Implement targeted retention strategies for customers showing signs of dissatisfaction (e.g., high support tickets).
  - Consider enhancing the value proposition for premium subscriptions to retain high-value customers.

## Conclusion
The analysis indicates that while overall churn rates are low, there are specific segments and regions that require attention. By focusing on customer engagement and addressing the needs of at-risk segments, the business can enhance customer retention and reduce churn rates effectively. 

### Next Steps
- Further investigate the isolated churn instances to understand underlying causes.
- Develop targeted marketing and retention strategies based on customer attributes and behaviors.
- Monitor churn rates regularly to assess the effectiveness of implemented strategies. 

This report serves as a foundational analysis for understanding customer churn dynamics and guiding strategic decisions to improve customer retention.

**Chain Summary (Detailed info in bambooai_consolidated_log.json file):**

| Metric                      | Value          |
|-----------------------------|----------------|
| Chain ID | 1776891758 |
| Total Prompt Tokens | 59248 |
| Total Completion Tokens | 13304 |
| Total Tokens | 72552 |
| Total Time (LLM Interact.) | 115.31 seconds |
| Average Response Speed | 115.37 tokens/second |
| Total Cost | $0.0510 |


 exit


Finished BambooAI conversation.
